# RAG - FAISS Integration

Full flow:

> Docs > Loader > Chunks > Embeddings > FAISS Index > Retriever > Top-K chunks > LLM > Answer

## Init

In [13]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

## Configs

In [9]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

## Flow

### Document Loader

In [3]:
file_path = '/Users/dunghq/Library/CloudStorage/GoogleDrive-dunghoangq@gmail.com/My Drive/0-Studies/algorithm/clrs/Introduction.to.Algorithms.4th.Edition.pdf'

loader = PyPDFLoader(file_path)
documents = loader.load()

### Chunking

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = splitter.split_documents(documents)

### Embedding

In [11]:
embedding_model = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12170.34it/s]


In [12]:
vectors = embedding_model.embed_documents(
    [doc.page_content for doc in chunks]
)

### FAISS Index

In [18]:
# M = 32 (links per vertex). Higher means more accurate but uses more RAM.
dimension = 384
def init_hnsw_index():
    index = faiss.IndexHNSWFlat(dimension, 16)
    index.hnsw.efConstruction = 64
    index.hnsw.efSearch = 16
    return index


In [20]:
# HNSW
# vector_stores = FAISS.from_documents(
#     documents=chunks,
#     embedding=embedding_model,
#     index_init_fn=init_hnsw_index
# )

# Flat
vector_stores = FAISS.from_documents(chunks, embedding_model)

### Retrieval

In [24]:
# Use docs = retriever.invoke("Explain Depth First Search")
retriever = vector_stores.as_retriever(search_kwargs={"k": 5})

### Prompt

LangChain builds something like

```
Context:

Chunk 1

...

Chunk 2

...

Chunk 3

Question:

Explain AVL trees.
```

So the prompt becomes:

```
You are an assistant.

Context:
[Chunk 1]

[Chunk 2]

[Chunk 3]

Question:

Explain Dijkstra's algorithm.
```

In [45]:
def generate_promt(retriever, question: str):
    prompt = ''''''
    docs = retriever.invoke(question)

    for doc in docs:
        prompt += "#"*10 + "\n"
        prompt += "Title: " + doc.metadata["title"] + " - Page " + str(doc.metadata["page"]) + "\n"
        prompt += doc.page_content + "\n"
    
    prompt += "#"*10 + "\n"
    prompt += "Question:" + "\n"
    prompt += question
    return prompt


### LLM

Send prompt to GPT > Answer.

## Tuning

- Use smaller number of chunks: k
- Smaller # characters per chunk
- Cross-encoder Reranking
- Compression
- Hierarchical retrieval

## Test

In [ ]:
docs = retriever.invoke("Explain Depth First Search")

for doc in docs:
    print("#"*80)
    print("Title:", doc.metadata["title"], "- Page", doc.metadata["page"])
    print(doc.page_content)

################################################################################
Title Introduction to Algorithms, Fourth Edition - Page 755
second depth-ﬁrst search and scan the vertices in order of increasing
ﬁnish times. Does this modiﬁed algorithm always produce correct
results?
################################################################################
Title Introduction to Algorithms, Fourth Edition - Page 737
discovery time u.d, and paint u gray. Lines 4–7 examine each vertex v
adjacent to u and recursively visit v if it is white. As line 4 considers each
vertex v ∈  Adj[u], the depth-ﬁrst search explores edge (u, v). Finally,
after every edge leaving u has been explored, lines 8–10 increment time,
record the ﬁnish time in u.f, and paint u black.
The results of depth-ﬁrst search may depend upon the order in which
line 5 of DFS examines the vertices and upon the order in which line 4
of DFS-VISIT visits the neighbors of a vertex. These different visitation
orders tend not to

In [46]:
prompt = generate_promt(retriever, "Explain Depth First Search")
print(prompt)

##########
Title: Introduction to Algorithms, Fourth Edition - Page 755
second depth-ﬁrst search and scan the vertices in order of increasing
ﬁnish times. Does this modiﬁed algorithm always produce correct
results?
##########
Title: Introduction to Algorithms, Fourth Edition - Page 737
discovery time u.d, and paint u gray. Lines 4–7 examine each vertex v
adjacent to u and recursively visit v if it is white. As line 4 considers each
vertex v ∈  Adj[u], the depth-ﬁrst search explores edge (u, v). Finally,
after every edge leaving u has been explored, lines 8–10 increment time,
record the ﬁnish time in u.f, and paint u black.
The results of depth-ﬁrst search may depend upon the order in which
line 5 of DFS examines the vertices and upon the order in which line 4
of DFS-VISIT visits the neighbors of a vertex. These different visitation
orders tend not to cause problems in practice, because many
applications of depth-ﬁrst search can use the result from any depth-ﬁrst
search.
##########
Titl